# 02 – Patterns and labels

Load the cached HI-Small transactions Parquet (`data/interim/hi_small_trans.parquet`).

In [60]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'data' / 'interim' / 'hi_small_trans.parquet'
DATA

PosixPath('/Users/ravinderkumar/Work/upskill/AI/AIAgent/mule-detection/data/interim/hi_small_trans.parquet')

In [61]:
df = pd.read_parquet(DATA)
print(f'{df.shape[0]:,} rows x {df.shape[1]} cols')
df.head()

5,078,345 rows x 13 cols


,timestamp,from_bank,from_account,to_bank,to_account,amount_received,receiving_currency,amount_paid,payment_currency,payment_format,is_laundering,src,dst
0,2022-09-01 00:20:00,10,8000EBD30,10,8000EBD30,"3,697.34",US Dollar,"3,697.34",US Dollar,Reinvestment,0,10_8000EBD30,10_8000EBD30
1,2022-09-01 00:20:00,3208,8000F4580,1,8000F5340,0.01,US Dollar,0.01,US Dollar,Cheque,0,3208_8000F4580,1_8000F5340
2,2022-09-01 00:00:00,3209,8000F4670,3209,8000F4670,"14,675.57",US Dollar,"14,675.57",US Dollar,Reinvestment,0,3209_8000F4670,3209_8000F4670
3,2022-09-01 00:02:00,12,8000F5030,12,8000F5030,"2,806.97",US Dollar,"2,806.97",US Dollar,Reinvestment,0,12_8000F5030,12_8000F5030
4,2022-09-01 00:06:00,10,8000F5200,10,8000F5200,"36,682.97",US Dollar,"36,682.97",US Dollar,Reinvestment,0,10_8000F5200,10_8000F5200


## Parse `HI-Small_Patterns.txt`

Each laundering attempt is a `BEGIN LAUNDERING ATTEMPT - <TYPE>[: <detail>]` ... `END LAUNDERING ATTEMPT - <TYPE>` block of headerless transaction rows. Parse into one row per transaction, tagged with `attempt_id` (0-based, file order), `pattern_type` and `pattern_detail`.

In [62]:
import re

PATTERNS = ROOT / 'data' / 'raw' / 'ibm_aml' / 'HI-Small_Patterns.txt'

TXN_COLS = [
    'timestamp', 'from_bank', 'from_account', 'to_bank', 'to_account',
    'amount_received', 'receiving_currency', 'amount_paid', 'payment_currency',
    'payment_format', 'is_laundering',
]
BEGIN_RE = re.compile(r'^BEGIN LAUNDERING ATTEMPT - (?P<type>[A-Z-]+)(?::\s*(?P<detail>.*))?$')
END_RE = re.compile(r'^END LAUNDERING ATTEMPT - (?P<type>[A-Z-]+)$')


def parse_patterns(path):
    """Return one row per transaction listed in the patterns file."""
    rows = []
    attempt_id, ptype, detail = -1, None, None
    for lineno, line in enumerate(Path(path).read_text().splitlines(), 1):
        line = line.strip()
        if not line:
            continue
        if m := BEGIN_RE.match(line):
            if ptype is not None:
                raise ValueError(f'line {lineno}: BEGIN inside open block')
            attempt_id += 1
            ptype, detail = m['type'], m['detail']
        elif m := END_RE.match(line):
            if m['type'] != ptype:
                raise ValueError(f'line {lineno}: END {m["type"]} closes {ptype}')
            ptype, detail = None, None
        else:
            if ptype is None:
                raise ValueError(f'line {lineno}: transaction outside a block')
            fields = line.split(',')
            if len(fields) != len(TXN_COLS):
                raise ValueError(f'line {lineno}: expected {len(TXN_COLS)} fields, got {len(fields)}')
            rows.append([attempt_id, ptype, detail, *fields])
    if ptype is not None:
        raise ValueError('file ended inside an open block')

    out = pd.DataFrame(rows, columns=['attempt_id', 'pattern_type', 'pattern_detail', *TXN_COLS])
    out['timestamp'] = pd.to_datetime(out['timestamp'], format='%Y/%m/%d %H:%M')
    out[['from_bank', 'to_bank']] = out[['from_bank', 'to_bank']].astype('int32')
    out[['amount_received', 'amount_paid']] = out[['amount_received', 'amount_paid']].astype(float)
    out['is_laundering'] = out['is_laundering'].astype('int8')
    out['src'] = out['from_bank'].astype(str) + '_' + out['from_account']
    out['dst'] = out['to_bank'].astype(str) + '_' + out['to_account']
    return out


pat = parse_patterns(PATTERNS)
print(f"{pat['attempt_id'].nunique()} attempts, {len(pat):,} transactions")
pat.head()

370 attempts, 3,209 transactions


,attempt_id,pattern_type,pattern_detail,timestamp,from_bank,from_account,to_bank,to_account,amount_received,receiving_currency,amount_paid,payment_currency,payment_format,is_laundering,src,dst
0,0,FAN-OUT,Max 16-degree Fan-Out,2022-09-01 00:06:00,21174,800737690,12,80011F990,"2,848.96",Euro,"2,848.96",Euro,ACH,1,21174_800737690,12_80011F990
1,0,FAN-OUT,Max 16-degree Fan-Out,2022-09-01 04:33:00,21174,800737690,20,80020C5B0,"8,630.40",Euro,"8,630.40",Euro,ACH,1,21174_800737690,20_80020C5B0
2,0,FAN-OUT,Max 16-degree Fan-Out,2022-09-01 09:14:00,21174,800737690,20,80006A5E0,"35,642.49",Yuan,"35,642.49",Yuan,ACH,1,21174_800737690,20_80006A5E0
3,0,FAN-OUT,Max 16-degree Fan-Out,2022-09-01 09:56:00,21174,800737690,220,8007A5B70,"5,738,987.96",US Dollar,"5,738,987.96",US Dollar,ACH,1,21174_800737690,220_8007A5B70
4,0,FAN-OUT,Max 16-degree Fan-Out,2022-09-01 11:28:00,21174,800737690,1244,80093C0D0,"7,254.53",US Dollar,"7,254.53",US Dollar,ACH,1,21174_800737690,1244_80093C0D0


In [63]:
attempts = pat.groupby('attempt_id').agg(
    pattern_type=('pattern_type', 'first'),
    pattern_detail=('pattern_detail', 'first'),
    n_txns=('src', 'size'),
    total_paid=('amount_paid', 'sum'),
)
accts = pat.melt(id_vars='attempt_id', value_vars=['src', 'dst'])
attempts['n_accounts'] = accts.groupby('attempt_id')['value'].nunique()

attempts.groupby('pattern_type').agg(
    attempts=('n_txns', 'size'), txns=('n_txns', 'sum'),
    median_txns=('n_txns', 'median'), median_accounts=('n_accounts', 'median'),
)

,attempts,txns,median_txns,median_accounts
pattern_type,,,,
BIPARTITE,49,263,4.00,8.00
CYCLE,54,287,4.00,4.00
FAN-IN,40,318,8.00,9.00
FAN-OUT,48,342,7.00,8.00
GATHER-SCATTER,51,716,14.00,15.00
RANDOM,41,191,3.00,4.00
SCATTER-GATHER,44,626,14.00,9.00
STACK,43,466,10.00,15.00


### Attempts and transactions per pattern type

In [64]:
counts = (
    pat.groupby('pattern_type')
    .agg(attempts=('attempt_id', 'nunique'), transactions=('attempt_id', 'size'))
    .sort_values('attempts', ascending=False)
)
counts.loc['TOTAL'] = counts.sum()
counts

,attempts,transactions
pattern_type,,
CYCLE,54,287
GATHER-SCATTER,51,716
BIPARTITE,49,263
FAN-OUT,48,342
SCATTER-GATHER,44,626
STACK,43,466
RANDOM,41,191
FAN-IN,40,318
TOTAL,370,3209


## Join pattern transactions to the main table

Match on timestamp, endpoints and both amounts (unique within the pattern file and within the laundering rows of the main table), then flag covered rows.

In [65]:
JOIN_KEY = ['timestamp', 'src', 'dst', 'amount_received', 'amount_paid']
assert not pat.duplicated(JOIN_KEY).any()  # pattern keys are unique

# attempt_id / pattern_type per main-table row (NaN = not in any known pattern)
df = df.merge(pat[JOIN_KEY + ['attempt_id', 'pattern_type']], on=JOIN_KEY, how='left', validate='many_to_one')
df['in_pattern'] = df['attempt_id'].notna()

print(f'pattern txns: {len(pat):,}  main-table rows matched: {df.in_pattern.sum():,}')
print('main-table label of matched rows:', df.loc[df.in_pattern, 'is_laundering'].value_counts().to_dict())
df.shape

pattern txns: 3,209  main-table rows matched: 3,209
main-table label of matched rows: {1: 3209}


(5078345, 16)

In [66]:
laund = df[df.is_laundering == 1]
covered = laund.in_pattern.sum()
print(f'laundering rows: {len(laund):,}')
print(f'covered by a known pattern: {covered:,} ({covered / len(laund):.1%})')
print(f'not covered: {len(laund) - covered:,} ({1 - covered / len(laund):.1%})')
print(f'pattern rows labelled 0 in main table: {((df.in_pattern) & (df.is_laundering == 0)).sum()}')

laundering rows: 5,177
covered by a known pattern: 3,209 (62.0%)
not covered: 1,968 (38.0%)
pattern rows labelled 0 in main table: 0


In [67]:
# who is in the uncovered laundering rows?
unc = laund[~laund.in_pattern]
pd.DataFrame({
    'covered': laund[laund.in_pattern].payment_format.value_counts(),
    'uncovered': unc.payment_format.value_counts(),
}).fillna(0).astype(int)

,covered,uncovered
payment_format,,
ACH,3208,1275
Bitcoin,1,55
Cash,0,108
Cheque,0,324
Credit Card,0,206


## Account → pattern types

An account is `bank_account` (as in `src` / `dst`). It takes part in an attempt if it sends or receives any of the attempt's transactions.

In [68]:
legs = pd.concat([
    pat[['src', 'attempt_id', 'pattern_type']].rename(columns={'src': 'account'}).assign(role='sender'),
    pat[['dst', 'attempt_id', 'pattern_type']].rename(columns={'dst': 'account'}).assign(role='receiver'),
])

# one row per (account, pattern type)
acct_pattern = (
    legs.groupby(['account', 'pattern_type'])
    .agg(attempts=('attempt_id', 'nunique'),
         as_sender=('role', lambda r: (r == 'sender').sum()),
         as_receiver=('role', lambda r: (r == 'receiver').sum()))
    .reset_index()
)

# one row per account
acct_types = (
    acct_pattern.groupby('account')
    .agg(pattern_types=('pattern_type', lambda s: sorted(s)),
         n_types=('pattern_type', 'size'),
         n_attempts=('attempts', 'sum'))
)
acct_types['n_attempts'] = legs.groupby('account')['attempt_id'].nunique()
print(f'{len(acct_types):,} accounts in known patterns')
acct_types.n_types.value_counts().sort_index().rename('accounts_by_n_types').to_frame()

3,170 accounts in known patterns


,accounts_by_n_types
n_types,
1,3078
2,50
3,12
4,10
5,5
6,4
7,4
8,7


In [69]:
# accounts in more than one pattern type, and the most common combinations
multi = acct_types[acct_types.n_types > 1]
print(f'{len(multi):,} accounts in >1 pattern type; {(acct_types.n_attempts > 1).sum():,} in >1 attempt')
multi.pattern_types.map(' + '.join).value_counts().head(10).to_frame('accounts')

92 accounts in >1 pattern type; 100 in >1 attempt


,accounts
pattern_types,
BIPARTITE + CYCLE + FAN-IN + FAN-OUT + GATHER-SCATTER + RANDOM + SCATTER-GATHER + STACK,7
GATHER-SCATTER + STACK,5
BIPARTITE + STACK,5
BIPARTITE + FAN-OUT + GATHER-SCATTER + RANDOM,4
CYCLE + FAN-IN + FAN-OUT + GATHER-SCATTER + RANDOM + SCATTER-GATHER + STACK,4
FAN-OUT + STACK,4
BIPARTITE + GATHER-SCATTER,3
CYCLE + STACK,3
RANDOM + STACK,3


In [70]:
# accounts per pattern type
acct_pattern.groupby('pattern_type').account.nunique().sort_values(ascending=False).to_frame('accounts')

,accounts
pattern_type,
GATHER-SCATTER,685
STACK,663
BIPARTITE,491
SCATTER-GATHER,369
FAN-OUT,359
FAN-IN,338
CYCLE,271
RANDOM,211


## Account-level mule label

`is_mule = 1` if the account sent **or** received at least one transaction with `is_laundering == 1` in the main table (all 5,177 laundering rows, not only those in known patterns).

In [71]:
sent = df.groupby('src').agg(n_sent=('is_laundering', 'size'), n_sent_laund=('is_laundering', 'sum'))
recv = df.groupby('dst').agg(n_recv=('is_laundering', 'size'), n_recv_laund=('is_laundering', 'sum'))
sent.index.name = recv.index.name = 'account'

accounts = sent.join(recv, how='outer').fillna(0).astype(int)
accounts['is_mule'] = ((accounts.n_sent_laund + accounts.n_recv_laund) > 0).astype('int8')
print(f'{len(accounts):,} accounts; {accounts.is_mule.sum():,} mules ({accounts.is_mule.mean():.3%})')
accounts.is_mule.value_counts().to_frame('accounts')

515,088 accounts; 6,357 mules (1.234%)


,accounts
is_mule,
0,508731
1,6357


In [72]:
# role of the mules: laundering money out, in, or both
mules = accounts[accounts.is_mule == 1]
role = pd.Series(
    ['both' if s and r else 'sender only' if s else 'receiver only'
     for s, r in zip(mules.n_sent_laund > 0, mules.n_recv_laund > 0)],
    index=mules.index,
)
role.value_counts().to_frame('mules')

,mules
receiver only,2981
sender only,2373
both,1003


In [73]:
# how many mules are visible through the known patterns?
accounts['in_pattern'] = accounts.index.isin(acct_types.index)
pd.crosstab(accounts.is_mule, accounts.in_pattern, rownames=['is_mule'], colnames=['in_pattern'])

in_pattern,False,True
is_mule,,
0,508731,0
1,3187,3170


## Mule roles

- **sender_only**: sent laundering txns, never received one
- **receiver_only**: received laundering txns, never sent one
- **pass_through**: did both (laundering money in and out)

In [74]:
has_out = accounts.n_sent_laund > 0
has_in = accounts.n_recv_laund > 0
accounts['mule_role'] = pd.Series(
    np.select([has_out & has_in, has_out, has_in], ['pass_through', 'sender_only', 'receiver_only'], default='not_mule'),
    index=accounts.index,
)
accounts['mule_role'].value_counts().to_frame('accounts')

,accounts
mule_role,
not_mule,508731
receiver_only,2981
sender_only,2373
pass_through,1003


In [75]:
# pass-through sanity check: does laundering money arrive before it leaves?
laund = df[df.is_laundering == 1]
first_in = laund.groupby('dst').timestamp.min().rename('first_in')
first_out = laund.groupby('src').timestamp.min().rename('first_out')
pt = accounts[accounts.mule_role == 'pass_through'].join(first_in).join(first_out)
print(f'pass-through accounts: {len(pt):,}')
print(f'  first laundering txn is incoming: {(pt.first_in < pt.first_out).mean():.1%}')
print(f'  first laundering txn is outgoing: {(pt.first_in > pt.first_out).mean():.1%}')
print(f'  same timestamp:                   {(pt.first_in == pt.first_out).mean():.1%}')

# role mix by pattern type (accounts in multiple types count once per type)
(acct_pattern.merge(accounts[['mule_role']], left_on='account', right_index=True)
 .pivot_table(index='pattern_type', columns='mule_role', values='account', aggfunc='nunique', fill_value=0))

pass-through accounts: 1,003
  first laundering txn is incoming: 85.9%
  first laundering txn is outgoing: 14.1%
  same timestamp:                   0.0%


mule_role,pass_through,receiver_only,sender_only
pattern_type,,,
BIPARTITE,49,219,223
CYCLE,271,0,0
FAN-IN,36,28,274
FAN-OUT,42,285,32
GATHER-SCATTER,75,297,313
RANDOM,163,25,23
SCATTER-GATHER,310,29,30
STACK,243,209,211


## Mule vs normal accounts

Per-account activity over all transactions (laundering and normal). Amounts are summed as raw numbers across currencies, so totals are only roughly comparable; medians and counts are the safer read.

In [76]:
out = df.groupby('src').agg(txns_out=('dst', 'size'), amount_out=('amount_paid', 'sum'),
                            cp_out=('dst', 'nunique'))
inn = df.groupby('dst').agg(txns_in=('src', 'size'), amount_in=('amount_received', 'sum'),
                            cp_in=('src', 'nunique'))
out.index.name = inn.index.name = 'account'

feat = accounts[['is_mule', 'mule_role']].join(out).join(inn).fillna(0)
cp = pd.concat([df[['src', 'dst']].set_axis(['account', 'cp'], axis=1),
                df[['dst', 'src']].set_axis(['account', 'cp'], axis=1)])
feat['counterparties'] = cp.drop_duplicates().groupby('account').size()
feat['txns'] = feat.txns_out + feat.txns_in
del cp
feat.shape

(515088, 10)

In [77]:
cols = ['txns_out', 'txns_in', 'amount_out', 'amount_in', 'cp_out', 'cp_in', 'counterparties']
feat.groupby('is_mule')[cols].median().rename(index={0: 'normal', 1: 'mule'}).T.assign(ratio=lambda t: t['mule'] / t['normal'])

is_mule,normal,mule,ratio
txns_out,2.00,4.00,2.00
txns_in,2.00,13.00,6.50
amount_out,"10,039.22","107,345.37",10.69
amount_in,"13,552.72","142,143.10",10.49
cp_out,1.00,2.00,2.00
cp_in,1.00,3.00,3.00
counterparties,2.00,5.00,2.50


In [78]:
feat.groupby('mule_role')[cols].median().T[['not_mule', 'sender_only', 'receiver_only', 'pass_through']]

mule_role,not_mule,sender_only,receiver_only,pass_through
txns_out,2.00,4.00,3.00,6.00
txns_in,2.00,6.00,16.00,16.00
amount_out,"10,039.22","305,741.54","28,369.50","291,147.74"
amount_in,"13,552.72","229,814.27","80,018.32","266,665.88"
cp_out,1.00,3.00,2.00,3.00
cp_in,1.00,2.00,4.00,4.00
counterparties,2.00,4.00,5.00,6.00


In [79]:
# share of accounts with no outgoing / no incoming txns, and heavy-tail check (mean vs median)
pd.DataFrame({
    'no_outgoing': feat.groupby('is_mule').apply(lambda g: (g.txns_out == 0).mean()),
    'no_incoming': feat.groupby('is_mule').apply(lambda g: (g.txns_in == 0).mean()),
    'mean_txns': feat.groupby('is_mule').txns.mean(),
    'p90_txns': feat.groupby('is_mule').txns.quantile(0.9),
}).rename(index={0: 'normal', 1: 'mule'})

,no_outgoing,no_incoming,mean_txns,p90_txns
is_mule,,,,
normal,0.04,0.19,18.68,51.00
mule,0.03,0.01,102.75,72.00


## Save account labels

In [80]:
OUT = ROOT / 'data' / 'processed' / 'account_labels.parquet'
OUT.parent.mkdir(parents=True, exist_ok=True)

labels = accounts[['is_mule', 'mule_role', 'in_pattern']].join(
    acct_types[['pattern_types', 'n_types', 'n_attempts']]
)
labels['pattern_types'] = labels['pattern_types'].map(lambda v: list(v) if isinstance(v, list) else [])
labels[['n_types', 'n_attempts']] = labels[['n_types', 'n_attempts']].fillna(0).astype('int16')
labels['mule_role'] = labels['mule_role'].astype('category')

labels.reset_index().to_parquet(OUT, index=False)
chk = pd.read_parquet(OUT)
print(f'saved {len(chk):,} accounts -> {OUT.relative_to(ROOT)}')
print(chk.dtypes.to_string())
chk.mule_role.value_counts().to_frame()

saved 515,088 accounts -> data/processed/account_labels.parquet
account               str
is_mule              int8
mule_role        category
in_pattern           bool
pattern_types      object
n_types             int16
n_attempts          int16


,count
mule_role,
not_mule,508731
receiver_only,2981
sender_only,2373
pass_through,1003


## Key findings

1. **The pattern file is a partial label source.** `HI-Small_Patterns.txt` holds 370 laundering attempts (3,209 transactions) in 8 types. Every one of them matches a row labelled laundering in the main table, but they cover only 3,209 of 5,177 laundering rows (**62.0%**).
2. **The uncovered 38% is not random.** All Cash, Cheque and Credit Card laundering rows (638) and most Bitcoin ones are outside the patterns, as are 1,275 ACH rows. The pattern file is almost entirely ACH (3,208 of 3,209 rows).
3. **Pattern sizes are uneven.** Attempt counts are similar across types (40 to 54), but GATHER-SCATTER and SCATTER-GATHER hold about 42% of pattern transactions (1,342). RANDOM (191) and BIPARTITE (263) are the thinnest.
4. **Mules are rare.** Defining a mule as an account that sent or received at least one laundering transaction gives **6,357 of 515,088 accounts (1.23%)**. Only half of them (3,170) appear in a known pattern, so pattern membership alone would miss 3,187 mules.
5. **Roles split three ways.** 2,981 are receiver-only, 2,373 sender-only and 1,003 pass-through. In 86% of pass-through accounts laundering money arrives before it leaves.
6. **Pattern type predicts role.** CYCLE accounts are all pass-through. FAN-IN accounts are mostly sender-only and FAN-OUT accounts mostly receiver-only. SCATTER-GATHER is mostly pass-through.
7. **Mules are busier and bigger.** Median mule vs normal: 13 vs 2 incoming transactions, 4 vs 2 outgoing, about 10x the money moved and 5 vs 2 counterparties. Nearly every mule receives money (1% never do vs 19% of normal accounts). Amounts mix currencies, so only the order of magnitude is reliable.
8. **Watch for leakage and shared accounts.** These activity gaps include the laundering transactions that define the label, so features should come from a window before laundering starts. 92 accounts sit in more than one pattern type (a few in all eight), so splits must group by account, not by transaction.
